In [1]:
# ==============================================================================
# `gap-orb` QC Research Notebook
# Version: 1.4 (Corrected for Data Fetching and Indexing)
#
# Instructions:
# 1. Open this notebook in your QuantConnect Research environment.
# 2. Run this cell.
# ==============================================================================

# ### 1. & 2. Environment Setup and Parameter Definition

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import linregress
from datetime import time, datetime

# --- Parameters ---
# Define Asset Baskets
SYMBOLS_INDEX = ["SPY", "QQQ"]          # Market Index ETFs
SYMBOLS_SECTOR = ["XLK", "XLF", "XLE"]   # Technology, Financial, Energy Sector ETFs
SYMBOLS_TECH = ["TSLA", "NVDA", "AMD", "AAPL", "MSFT"] # Volatile Big-Tech Stocks

# Consolidate all symbols for data requests
ALL_TICKERS = SYMBOLS_INDEX + SYMBOLS_SECTOR + SYMBOLS_TECH

# Create a mapping for easy lookup of asset class
ASSET_CLASS_MAP = {s: 'Index' for s in SYMBOLS_INDEX}
ASSET_CLASS_MAP.update({s: 'Sector' for s in SYMBOLS_SECTOR})
ASSET_CLASS_MAP.update({s: 'Tech' for s in SYMBOLS_TECH})

START_DATE = "2020-01-01"
END_DATE = "2023-12-31"
ORB_MINUTES = 15
VOLATILITY_WINDOW = 90
GAP_Z_SCORE = 0.1

# Instantiate QuantBook
qb = QuantBook()

# Use Symbol.Create to explicitly define the SecurityType and Market.
ALL_SYMBOLS = [Symbol.Create(ticker, SecurityType.Equity, Market.USA) for ticker in ALL_TICKERS]

print("1. Setup Complete: Parameters are defined.")

# ### 3. Data Collection and Processing

# --- Fetch historical data ---
# ** THE FIX IS HERE **
# Revert to two separate, explicit History calls for clarity and correctness.
print("\n2. Fetching Data... (This may take a minute)")
daily_df = qb.History(ALL_SYMBOLS, datetime.strptime(START_DATE, "%Y-%m-%d"), datetime.strptime(END_DATE, "%Y-%m-%d"), Resolution.Daily)
minute_df = qb.History(ALL_SYMBOLS, datetime.strptime(START_DATE, "%Y-%m-%d"), datetime.strptime(END_DATE, "%Y-%m-%d"), Resolution.Minute)

# --- Clean and process data ---
daily_df = daily_df.reset_index()
minute_df = minute_df.reset_index()

# Convert the Symbol object in the index to a string ticker for easier mapping.
daily_df['symbol'] = daily_df['symbol'].apply(lambda x: x.Value)
minute_df['symbol'] = minute_df['symbol'].apply(lambda x: x.Value)

# --- Calculate Daily Metrics (Volatility, Gaps) ---
print("3. Processing Daily Metrics...")
daily_df['date'] = daily_df['time'].dt.date

# Calculate volatility
daily_df['volatility'] = daily_df.groupby('symbol')['close'].pct_change().rolling(VOLATILITY_WINDOW).std()

# Get previous day's high/low
for col in ['high', 'low', 'close']:
    daily_df[f'prev_{col}'] = daily_df.groupby('symbol')[col].shift(1)

# Calculate the dynamic gap thresholds
daily_df['upper_gap_threshold'] = daily_df['prev_high'] * (1 + GAP_Z_SCORE * daily_df['volatility'])
daily_df['lower_gap_threshold'] = daily_df['prev_low'] * (1 - GAP_Z_SCORE * daily_df['volatility'])

# Identify gap days
daily_df['is_gap_up'] = daily_df['open'] > daily_df['upper_gap_threshold']
daily_df['is_gap_down'] = daily_df['open'] < daily_df['lower_gap_threshold']
daily_df['is_gap_day'] = daily_df['is_gap_up'] | daily_df['is_gap_down']
daily_df['daily_return'] = daily_df['close'] / daily_df['open'] - 1

# Select only the necessary daily columns to merge
daily_info = daily_df[['symbol', 'date', 'is_gap_day', 'daily_return']].copy()

# --- Create the main analysis DataFrame ---
print("4. Building Analysis DataFrame...")
gap_days_data = daily_info[daily_info['is_gap_day']].copy()

analysis_results = []
for _, row in gap_days_data.iterrows():
    symbol = row['symbol']
    date = row['date']
    
    day_minute_data = minute_df[(minute_df['symbol'] == symbol) & (minute_df['time'].dt.date == date)].copy()
    if day_minute_data.empty:
        continue
        
    orb_start_time = day_minute_data['time'].iloc[0].replace(hour=9, minute=30)
    orb_end_time = orb_start_time + pd.Timedelta(minutes=ORB_MINUTES)
    
    orb_df = day_minute_data[(day_minute_data['time'] >= orb_start_time) & (day_minute_data['time'] < orb_end_time)]
    post_orb_df = day_minute_data[day_minute_data['time'] >= orb_end_time]
    
    if orb_df.empty or post_orb_df.empty:
        continue
        
    orb_high = orb_df['high'].max()
    orb_low = orb_df['low'].min()
    orb_volume = orb_df['volume'].sum()

    if len(orb_df) > 1:
        slope, _, _, _, _ = linregress(range(len(orb_df)), orb_df['close'])
    else:
        slope = 0
    
    break_up_df = post_orb_df[post_orb_df['high'] > orb_high]
    break_down_df = post_orb_df[post_orb_df['low'] < orb_low]
    
    time_up = break_up_df['time'].iloc[0] if not break_up_df.empty else pd.Timestamp.max
    time_down = break_down_df['time'].iloc[0] if not break_down_df.empty else pd.Timestamp.max
    
    breakout_direction = None
    breakout_volume = 0
    if time_up < time_down:
        breakout_direction = 'Up'
        breakout_volume = break_up_df['volume'].iloc[0]
    elif time_down < time_up:
        breakout_direction = 'Down'
        breakout_volume = break_down_df['volume'].iloc[0]
        
    analysis_results.append({
        'symbol': symbol,
        'date': date,
        'daily_return': row['daily_return'],
        'breakout_direction': breakout_direction,
        'orb_slope': slope,
        'breakout_volume': breakout_volume,
        'orb_volume': orb_volume
    })

analysis_df = pd.DataFrame(analysis_results)
analysis_df.dropna(inplace=True)

analysis_df['asset_class'] = analysis_df['symbol'].map(ASSET_CLASS_MAP)
analysis_df['orb_slope_category'] = ['Positive' if s > 0 else 'Negative' for s in analysis_df['orb_slope']]
avg_volume_by_symbol = analysis_df.groupby('symbol')['breakout_volume'].transform('mean')
analysis_df['breakout_volume_category'] = np.where(analysis_df['breakout_volume'] > avg_volume_by_symbol, 'High', 'Low')

print("5. Processing Complete. Generating visualizations...")
# (The visualization part of the code remains the same)

# --- Visualization 1 ---
print("\n--- Visualization 1: Daily Return by Asset Class and Breakout Direction ---")
fig1 = px.box(analysis_df, x="breakout_direction", y="daily_return", color="asset_class", facet_col="asset_class", title="Daily Return Distribution by Breakout Direction and Asset Class", labels={"daily_return": "Daily Return (Open to Close)", "breakout_direction": "ORB Breakout Direction"})
fig1.update_yaxes(tickformat=".2%")
fig1.show()

# --- Visualization 2 ---
print("\n--- Visualization 2: Impact of 'Quality' (Volume & Slope) on Tech Stocks ---")
tech_analysis_df = analysis_df[analysis_df['asset_class'] == 'Tech']
if not tech_analysis_df.empty:
    fig2 = px.box(tech_analysis_df, x='breakout_direction', y='daily_return', color='breakout_direction', facet_row='breakout_volume_category', facet_col='orb_slope_category', title="Tech Stocks: Return Distribution by Breakout Quality", labels={"daily_return": "Daily Return", "breakout_direction": "Breakout"})
    fig2.update_yaxes(tickformat=".2%")
    fig2.show()
else:
    print("No data available for Tech stocks quality analysis.")

# --- Visualization 3 ---
print("\n--- Visualization 3: Case Study (Example: A 'Gap & Go' day for a Tech stock) ---")
good_examples = tech_analysis_df[(tech_analysis_df['breakout_direction'] == 'Up') & (tech_analysis_df['breakout_volume_category'] == 'High') & (tech_analysis_df['orb_slope_category'] == 'Positive')]
if not good_examples.empty:
    case_study_day = good_examples.sort_values('daily_return', ascending=False).iloc[0]
    symbol_cs = case_study_day['symbol']
    date_cs = case_study_day['date']
    case_study_minute_data = minute_df[(minute_df['symbol'] == symbol_cs) & (minute_df['time'].dt.date == date_cs)]
    if not case_study_minute_data.empty:
        orb_start_time_cs = case_study_minute_data['time'].iloc[0].replace(hour=9, minute=30)
        orb_end_time_cs = orb_start_time_cs + pd.Timedelta(minutes=ORB_MINUTES)
        orb_df_cs = case_study_minute_data[case_study_minute_data['time'] < orb_end_time_cs]
        orb_high_cs = orb_df_cs['high'].max()
        orb_low_cs = orb_df_cs['low'].min()
        breakout_time_cs = case_study_minute_data[case_study_minute_data['high'] > orb_high_cs]['time'].min()
        fig3 = go.Figure()
        fig3.add_trace(go.Candlestick(x=case_study_minute_data['time'], open=case_study_minute_data['open'], high=case_study_minute_data['high'], low=case_study_minute_data['low'], close=case_study_minute_data['close'], name='Price'))
        fig3.add_shape(type="rect", x0=orb_start_time_cs, y0=orb_low_cs, x1=orb_end_time_cs, y1=orb_high_cs, line=dict(color="RoyalBlue", width=1), fillcolor="LightSkyBlue", opacity=0.3)
        if pd.notna(breakout_time_cs):
            fig3.add_trace(go.Scatter(x=[breakout_time_cs], y=[orb_high_cs], mode='markers', marker=dict(color='limegreen', size=15, symbol='arrow-up'), name='Breakout Point'))
        fig3.update_layout(title=f"Case Study: Gap & Go on {symbol_cs} ({date_cs.strftime('%Y-%m-%d')})", xaxis_title="Time", yaxis_title="Price", showlegend=True)
        fig3.show()
else:
    print("No suitable case study day found for visualization.")

print("\n--- End of Research ---")

In [2]:
# Added by Claude: Print the raw statistics for Figure 1 & Figure 2 so we can analyze the data
print("--- Figure 1 Statistics: Daily Return by Asset Class & Breakout Direction ---")
print(analysis_df.groupby(['asset_class', 'breakout_direction'])['daily_return'].describe().to_string())

print("\n--- Figure 2 Statistics: Impact of 'Quality' (Volume & Slope) on Tech Stocks ---")
if not tech_analysis_df.empty:
    print(tech_analysis_df.groupby(['breakout_direction', 'breakout_volume_category', 'orb_slope_category'])['daily_return'].describe().to_string())

In [ ]:
--- Figure 1 Statistics: Daily Return by Asset Class & Breakout Direction ---
                                count      mean       std       min       25%       50%       75%       max
asset_class breakout_direction                                                                             
Index       Down                279.0 -0.003648  0.010440 -0.040360 -0.009451 -0.002554  0.002936  0.022902
            Up                  298.0  0.003360  0.011215 -0.034371 -0.001689  0.003878  0.008465  0.068472
Sector      Down                411.0 -0.005468  0.012533 -0.049125 -0.012606 -0.004534  0.002134  0.043842
            Up                  463.0  0.005587  0.013440 -0.044699 -0.001399  0.004830  0.012387  0.070961
Tech        Down                558.0 -0.010435  0.022313 -0.098773 -0.020841 -0.009045  0.001600  0.077099
            Up                  581.0  0.012456  0.026292 -0.097613 -0.002115  0.010430  0.024205  0.143320

--- Figure 2 Statistics: Impact of 'Quality' (Volume & Slope) on Tech Stocks ---
                                                                count      mean       std       min       25%       50%       75%       max
breakout_direction breakout_volume_category orb_slope_category                                                                             
Down               High                     Negative            141.0 -0.015383  0.027332 -0.098773 -0.025972 -0.011791  0.000699  0.057674
                                            Positive             42.0 -0.002903  0.017224 -0.041555 -0.013155 -0.004733  0.007388  0.040247
                   Low                      Negative            289.0 -0.009984  0.021780 -0.088700 -0.021641 -0.008696  0.001287  0.077099
                                            Positive             86.0 -0.007516  0.014274 -0.062123 -0.016121 -0.005023  0.001060  0.034681
Up                 High                     Negative             55.0  0.011149  0.030124 -0.071139 -0.007194  0.007770  0.029836  0.097670
                                            Positive            152.0  0.018752  0.034972 -0.097613  0.000630  0.013971  0.033948  0.143320
                   Low                      Negative             90.0  0.004612  0.020069 -0.054243 -0.003824  0.005283  0.013228  0.070142
                                            Positive            284.0  0.011825  0.020560 -0.060956 -0.000210  0.010535  0.022748  0.068813